In [2]:
# 运行此单元格导入必要的库
import pandas as pd
import numpy as np
import cv2
import re
import jieba
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from autograder import *

# 忽略警告信息
import warnings
warnings.filterwarnings('ignore')
print("环境初始化完成！")

环境初始化完成！


In [16]:
# 读取 Kaggle House Prices 数据集
df = pd.read_csv('data\\train.csv')
print(f"原始数据维度: {df.shape}")

# 为了练习，我们人为制造一些重复行和特定的缺失值场景
df = pd.concat([df, df.iloc[:5]], ignore_index=True)
df.loc[10:15, 'LotFrontage'] = np.nan
print(f"注入噪声后的数据维度: {df.shape}")

原始数据维度: (1460, 81)
注入噪声后的数据维度: (1465, 81)


In [21]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
def handle_missing_and_duplicates(data:pd.DataFrame):
    """
    处理 DataFrame 中的缺失值和重复值
    """
    df_clean = data.copy()
    
    # ================================================================ #
    # TODO: 1. 删除 df_clean 中的完全重复的行
    #       2. 对于数值型特征的缺失值，使用该列的中位数进行填充
    #       3. 对于类别型特征（object）的缺失值，使用字符串 "Missing" 填充
    # ================================================================ #
    # 易错：这里不应该写为赋值形式
    df_clean.drop_duplicates(inplace=True)
    # 这个判断类型的可以记一下
    from pandas.api.types import is_numeric_dtype, is_object_dtype
    for col in df_clean.columns:
        if col == 'Id':
            continue
        
        if is_numeric_dtype(df_clean[col]):
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
        elif is_object_dtype(df_clean[col]):
            df_clean[col].fillna("Missing", inplace=True)
    # ================================================================ #
    # END YOUR CODE
    # ================================================================ #
    return df_clean

df_cleaned = handle_missing_and_duplicates(df)
check_missing_and_duplicates(df_cleaned)

✅ 缺失值与重复值处理通过！


In [31]:
def handle_outliers(data:pd.DataFrame, column):
    """
    使用 IQR (四分位距) 方法检测并剔除指定列的异常值
    """
    df_out = data.copy()
    
    # ================================================================ #
    # TODO: 1. 计算 column 列的 Q1 (25%) 和 Q3 (75%)
    #       2. 计算 IQR = Q3 - Q1
    #       3. 定义上下界：下界 = Q1 - 1.5 * IQR, 上界 = Q3 + 1.5 * IQR
    #       4. 过滤掉 df_out 中该 column 值不在 [下界, 上界] 范围内的行
    # ================================================================ #
    q1 = df_out[column].quantile(0.25)
    q3 = df_out[column].quantile(0.75)
    IQR = q3 - q1
    lower = q1 - 1.5*IQR
    upper = q3 + 1.5*IQR
    # 过滤的方法是直接取索引
    df_out = df_out[(df_out[column] >= lower) & (df_out[column] <= upper)]
    
    # ================================================================ #
    # END YOUR CODE
    # ================================================================ #
    return df_out

# 以居住面积 (GrLivArea) 为例处理异常值
df_no_outliers = handle_outliers(df_cleaned, 'GrLivArea')
check_outliers(df_cleaned, df_no_outliers)

✅ 异常值过滤通过！原数据量: 1460, 清洗后数据量: 1429


In [35]:
# 提取部分连续数值型特征用于练习
num_features = ['LotArea', 'GrLivArea', 'TotalBsmtSF', 'SalePrice']
X_num = df_no_outliers[num_features].values

def scale_features(X, method='zscore'):
    """
    根据 method 选用归一化或标准化
    """
    X_scaled = None
    # ================================================================ #
    # TODO: 使用 sklearn 的 StandardScaler (当 method=='zscore'时) 
    #       或 MinMaxScaler (当 method=='minmax'时) 对 X 进行转换
    # ================================================================ #
    if method == 'zscore':
        X_scaled = StandardScaler().fit_transform(X=X)
    elif method=='minmax':
        X_scaled = MinMaxScaler().fit_transform(X=X)
    # ================================================================ #
    # END YOUR CODE
    # ================================================================ #
    return X_scaled

X_zscore = scale_features(X_num, method='zscore')
check_scaling(X_zscore, method='zscore')

X_minmax = scale_features(X_num, method='minmax')
check_scaling(X_minmax, method='minmax')

✅ Z-Score 标准化结果正确！
✅ Min-Max 归一化结果正确！


In [36]:
def apply_pca(X, n_components):
    """
    对标准化后的数据执行主成分分析
    """
    X_pca = None
    # ================================================================ #
    # TODO: 初始化 PCA 并保留 n_components 个主成分，对 X 进行拟合和转换
    # ================================================================ #
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X)
    # ================================================================ #
    # END YOUR CODE
    # ================================================================ #
    return X_pca

# 我们将前面 Z-score 标准化后的 4 维特征降至 2 维
X_pca_result = apply_pca(X_zscore, n_components=2)
check_pca(X_pca_result, expected_dim=2)

✅ PCA 降维通过！数据已降至 2 维。


In [39]:
# 包含特写领域知识的非结构化中文文本语料
raw_texts = [
    "【研究突破】基于LLM与RAG的中文仇恨言论四元组抽取任务，准确率已达95.5%！！！@AI_Lab #NLP https://t.co/xyz",
    "【前沿探索】转向具身智能，五指灵巧手（dexterous manipulation）的序列预测模型实验验证成功... \n详细代码见Github！"
]

stop_words = {'的', '了', '与', '基于', '在', '是', '已', '见'}

def text_pipeline(texts):
    """
    完整文本处理管线
    """
    cleaned_docs = []
    
    for text in texts:
        # ================================================================ #
        # TODO: 1. 正则去噪：剔除 URL、@用户、#话题、以及特殊标点符号（如【】！等），保留中英文字符和数字
        #       2. jieba 分词：对清洗后的文本进行精确模式分词
        #       3. 去停用词：过滤掉在 stop_words 集合中的词
        #       4. 将过滤后的词语用空格拼接成字符串，存入 cleaned_docs 列表
        # ================================================================ #
        cleaned_text = re.sub(r'https?://\S+|www\.\S+|@\S+|#\S+|【|】|！', '', text)
        tokens = jieba.cut(cleaned_text, cut_all=False)
        res = ""
        for token in tokens:
            if token in stop_words:
                continue
            res += token
        cleaned_docs.append(res)
        # ================================================================ #
        # END YOUR CODE
        # ================================================================ #
        
    # ================================================================ #
    # TODO: 5. 使用 TfidfVectorizer 将 cleaned_docs 转换为 TF-IDF 矩阵
    #       获取 tfidf_matrix (特征矩阵) 和 vocab (词汇表字典, .vocabulary_)
    # ================================================================ #
    tf = TfidfVectorizer().fit(cleaned_docs)
    tfidf_matrix = tf.transform(cleaned_docs)
    vocab = tf.vocabulary_
    # ================================================================ #
    # END YOUR CODE
    # ================================================================ #
    return tfidf_matrix, vocab

tfidf_mat, vocabulary = text_pipeline(raw_texts)
check_text_pipeline(tfidf_mat, vocabulary)

✅ 文本清理、分词与 TF-IDF 提取通过！共提取到 8 个特征词。
特征词表示例: ['研究突破llmrag中文仇恨言论四元组抽取任务', '准确率已达95', '前沿探索转向具身智能', '五指灵巧手', 'dexterous']


In [42]:
# 生成一张随机的伪彩色图像模拟输入 (高度 300, 宽度 400, 3通道 BGR 格式)
dummy_image_bgr = np.random.randint(0, 256, (300, 400, 3), dtype=np.uint8)

def image_pipeline(img_bgr):
    """
    图像标准化预处理管线
    """
    img_processed = None
    # ================================================================ #
    # TODO: 1. 颜色空间转换：将图像从 BGR 转换到 RGB 空间 (提示: cv2.cvtColor)
    #       2. 尺寸统一：将图像 Resize 为 224x224 分辨率
    #       3. 像素插值：在 Resize 时，指定使用双线性插值 (cv2.INTER_LINEAR)
    # ================================================================ #
    img_processed = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_processed = cv2.resize(img_processed, (224, 224), interpolation=cv2.INTER_LINEAR)
    # ================================================================ #
    # END YOUR CODE
    # ================================================================ #
    return img_processed

processed_img = image_pipeline(dummy_image_bgr)
check_image_pipeline(processed_img)

✅ 图像尺寸统一及颜色空间转换处理通过！(Shape: 224x224x3)
